# 02 选股示例

用 quant.filter 的过滤链从全市场筛股票。每个过滤器把快照表变窄一步，
过滤链会记录每一步剩了多少只。

说明：min_listing / price_change / ma_trend 等需要读取历史行情，首次调用会慢一些
（之后在同一次运行里走缓存）。

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from quant.filter import FilterChain, FilterContext, describe_filters

pd.set_option("display.width", 200)
for name, description in describe_filters():
    print("%-16s %s" % (name, description))

## 用字符串描述构造过滤链

每一步格式：name 或 name:k=v,k=v。

In [ ]:
specs = [
    "exclude_st",                                  # 去掉 ST / 退市
    "min_listing:bars=250",                        # 上市满一年
    "market_cap:max=200",                          # 市值 < 200 亿
    "pe:min=0,max=30",                             # PE(TTM) 0~30
    "pe_percentile:window_years=10,max=20",        # PE 处于近 10 年最低 20% 分位
    "pb_percentile:window_years=10,max=30",        # PB 同样处于历史低位
    "roe:min=8",                                   # ROE >= 8%
    "break_ma:period=5,mode=cross",                # 当日突破 5 日均线
]

import time
t0 = time.time()
with FilterContext() as ctx:
    result = FilterChain.from_specs(specs).run(ctx)
print("用时 %.1fs" % (time.time() - t0))
print(result.report())

In [ ]:
cols = ["symbol", "name", "close_org", "market_cap", "pe", "pe_pct_10y",
        "pb", "pb_pct_10y", "ps", "roe"]
cols = [c for c in cols if c in result.frame.columns]
result.frame[cols].sort_values("pe_pct_10y").head(20)

## 用对象构造（适合在 notebook 里反复调参）

In [ ]:
from quant.filter.builtin import DebtRatio, MarketCap, Pb, Roe

custom = FilterChain([MarketCap(max=500), Pb(max=3), Roe(min=8), DebtRatio(max=60)])

with FilterContext() as ctx:
    custom_result = custom.run(ctx)

print(custom_result.report())
custom_result.frame[["symbol", "name", "pb", "roe", "debt_ratio", "market_cap"]].head(10)

## 自定义过滤器

实现 Filter 协议并注册，就能在命令行里用字符串引用。

In [ ]:
from quant.filter import register

@register
class SmallCap:
    name = "small_cap"
    description = "市值小于 80 亿"
    def __init__(self, max_yi=80):
        self.max_yi = max_yi
    def apply(self, df, ctx):
        return df[df["market_cap"] <= self.max_yi * 1e8]

with FilterContext() as ctx:
    r = FilterChain.from_specs(["small_cap", "exclude_st"]).run(ctx)
print(r.report())
r.frame[["symbol", "name", "market_cap"]].head()

## 导出

命令行等价形式：

    quant screen --filter exclude_st --filter market_cap:max=200 --filter growth:min=20 --out picks.csv

In [ ]:
result.frame.to_csv("picks.csv", index=False, encoding="utf_8_sig")
print("已导出 picks.csv:", len(result.frame), "行")